In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

awsaf49_brats20_dataset_training_validation_path = kagglehub.dataset_download('awsaf49/brats20-dataset-training-validation')
abdurrahman4127_model_weight_path = kagglehub.dataset_download('abdurrahman4127/model-weight')
abdurrahman4127_metal_b_k_path = kagglehub.notebook_output_download('abdurrahman4127/metal-b-k')

print('Data source import complete.')


# IMPORTS

In [ ]:
!pip install -U monai
!pip install -q nibabel

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from typing import List, Dict, Sequence, Optional, Tuple
import random
import os
import numpy as np
from tqdm import tqdm
from statistics import mean
import copy

from tqdm import tqdm
import logging

import matplotlib.pyplot as plt

try:
    import nibabel as nib
except ImportError:
    print("ERROR: nibabel library not found. Please install it: pip install nibabel")
    nib = None

# Using MONAI for established deep learning components for medical imaging
# https://monai.io/
from torch.optim import Adam
from monai.losses import DiceLoss, FocalLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.losses import DiceCELoss, DiceLoss, FocalLoss

# Using PyTorch's Automatic Mixed Precision for performance
from torch.cuda.amp import GradScaler
import torch.amp # For autocast
from statistics import mean

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, jaccard_score, accuracy_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 40.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 20.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninst

<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2025-08-06 08:56:22.599362: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754470582.820503      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754470582.880441      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# MAIN MODEL

In [ ]:
ALL_MODALITY_NAMES_DEFAULT = ["flair", "t1", "t1ce", "t2"]

def _preprocess_spatial_utility(
    volume: np.ndarray,
    target_shape: Tuple[int, int, int] = (128, 128, 128)
) -> np.ndarray:
    """
    center crop or pad 3d volume to target shape.
    
    args:
        volume: 3d input array of shape (d, h, w)
        target_shape: desired output shape (d, h, w)
    
    returns:
        preprocessed volume of target shape
    """
    current_shape = volume.shape
    result = np.zeros(target_shape, dtype=volume.dtype)
    
    # center crop if current larger than target
    slices_src = []
    slices_dst = []
    for i, (current, target) in enumerate(zip(current_shape, target_shape)):
        if current >= target:
            start = (current - target) // 2
            slices_src.append(slice(start, start + target))
            slices_dst.append(slice(0, target))
        else:
            start = (target - current) // 2
            slices_src.append(slice(0, current))
            slices_dst.append(slice(start, start + current))
    
    result[tuple(slices_dst)] = volume[tuple(slices_src)]
    return result

def _create_multilabel_seg_masks_utility(
    seg_array: np.ndarray
) -> np.ndarray:
    """
    convert brats integer segmentation labels to 3-channel binary masks.
    
    channel 0: whole tumor (label >= 1)
    channel 1: tumor core (label 1 or 3)
    channel 2: enhancing tumor (label 4)
    
    args:
        seg_array: segmentation array with labels [0, 1, 2, 3, 4]
    
    returns:
        binary mask array of shape (3, d, h, w)
    """
    wt = (seg_array >= 1).astype(np.float32)
    tc = (
        ((seg_array == 1) | (seg_array == 3)).astype(np.float32)
    )
    et = (seg_array == 4).astype(np.float32)
    
    return np.stack([wt, tc, et], axis=0)

def _preprocess_single_nifti_for_eval_utility(
    modality_path: str,
    target_shape: Tuple[int, int, int] = (128, 128, 128)
) -> Tuple[torch.Tensor, np.ndarray]:
    """
    load and preprocess single nifti file for inference.
    
    args:
        modality_path: path to nifti file
        target_shape: target spatial dimensions
    
    returns:
        tuple of preprocessed tensor and original volume info
    """
    nifti_obj = nib.load(modality_path)
    volume = np.asarray(nifti_obj.dataobj)
    
    # preprocess
    volume_processed = _preprocess_spatial_utility(
        volume,
        target_shape=target_shape
    )
    
    # normalize
    v_min = np.min(volume_processed)
    v_max = np.max(volume_processed)
    if v_max > v_min:
        volume_normalized = (
            (volume_processed - v_min) / (v_max - v_min)
        )
    else:
        volume_normalized = volume_processed
    
    # to tensor
    tensor = torch.from_numpy(
        volume_normalized[np.newaxis, ...]
    ).float()
    
    return tensor, volume

# nnUNet ARCHITECTURE COMPONENTS
class nnUNetConvBlock(nn.Module):
    """
    double 3d convolution with group normalization and leaky relu.
    
    implements the basic building block: conv3d -> groupnorm ->
    leakyrelu -> conv3d -> groupnorm -> leakyrelu + optional dropout.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        dropout_rate: float = 0.0
    ):
        """
        args:
            in_channels: number of input channels
            out_channels: number of output channels
            kernel_size: convolution kernel size
            dropout_rate: dropout probability
        """
        super().__init__()
        padding = (kernel_size - 1) // 2
        
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding
        )
        self.norm1 = nn.GroupNorm(
            num_groups=min(32, out_channels),
            num_channels=out_channels
        )
        
        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding
        )
        self.norm2 = nn.GroupNorm(
            num_groups=min(32, out_channels),
            num_channels=out_channels
        )
        
        self.dropout = nn.Dropout3d(p=dropout_rate) if dropout_rate > 0 else None
        self.activation = nn.LeakyReLU(negative_slope=0.01, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        forward pass through double conv block.
        
        args:
            x: input tensor of shape (b, c_in, d, h, w)
        
        returns:
            output tensor of shape (b, c_out, d, h, w)
        """
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.activation(x)
        
        if self.dropout:
            x = self.dropout(x)
        
        x = self.conv2(x)
        x = self.norm2(x)
        x = self.activation(x)
        
        return x

class nnUNetEncoder(nn.Module):
    """
    5-level hierarchical encoder for 3d medical images.
    
    progressively downsamples with max pooling and processes with
    conv blocks, returning skip connections at each level.
    """
    def __init__(
        self,
        in_channels: int = 1,
        base_features: int = 32,
        dropout_rate: float = 0.0
    ):
        """
        args:
            in_channels: number of input channels
            base_features: base feature dimension (doubled at each level)
            dropout_rate: dropout probability for conv blocks
        """
        super().__init__()
        self.in_channels = in_channels
        self.base_features = base_features
        
        # feature dimensions at each level
        self.level_features = [
            base_features,
            base_features * 2,
            base_features * 4,
            base_features * 8,
            base_features * 16
        ]
        
        # encoder levels
        self.conv_blocks = nn.ModuleList()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        
        in_feat = in_channels
        for out_feat in self.level_features:
            self.conv_blocks.append(
                nnUNetConvBlock(
                    in_feat,
                    out_feat,
                    kernel_size=3,
                    dropout_rate=dropout_rate
                )
            )
            in_feat = out_feat

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        """
        encode input through 5 levels, return skip connections.
        
        args:
            x: input tensor of shape (b, c_in, d, h, w)
        
        returns:
            list of 5 skip connection tensors at each level
        """
        skips = []
        
        for level in range(5):
            x = self.conv_blocks[level](x)
            skips.append(x)
            
            if level < 4:
                x = self.pool(x)
        
        return skips

class nnUNetDecoder(nn.Module):
    """
    symmetric decoder that upsamples and processes features.
    
    concatenates skip connections from encoder and produces
    final segmentation output.
    """
    def __init__(
        self,
        encoder_feature_dims: List[int],
        out_channels: int = 3,
        dropout_rate: float = 0.0
    ):
        """
        args:
            encoder_feature_dims: list of encoder feature dimensions
            out_channels: number of output segmentation channels
            dropout_rate: dropout probability
        """
        super().__init__()
        self.encoder_feature_dims = encoder_feature_dims
        self.out_channels = out_channels
        
        # upsampling layers
        self.upsample = nn.Upsample(
            scale_factor=2,
            mode='trilinear',
            align_corners=False
        )
        
        # decoder conv blocks (reversed from encoder)
        self.conv_blocks = nn.ModuleList()
        for i in range(4, 0, -1):
            # concatenate skip + upsampled, process
            in_feat = encoder_feature_dims[i] + encoder_feature_dims[i - 1]
            out_feat = encoder_feature_dims[i - 1]
            
            self.conv_blocks.append(
                nnUNetConvBlock(
                    in_feat,
                    out_feat,
                    kernel_size=3,
                    dropout_rate=dropout_rate
                )
            )
        
        # final output layer
        self.final_conv = nn.Conv3d(
            encoder_feature_dims[0],
            out_channels,
            kernel_size=1
        )

    def forward(self, skips: List[torch.Tensor]) -> torch.Tensor:
        """
        decode skip connections to segmentation output.
        
        args:
            skips: list of 5 skip connection tensors
        
        returns:
            segmentation logits of shape (b, out_channels, d, h, w)
        """
        # start from bottleneck (skip[4])
        x = skips[4]
        
        # decode levels 4 -> 0
        for level in range(4, 0, -1):
            x = self.upsample(x)
            x = torch.cat([x, skips[level - 1]], dim=1)
            x = self.conv_blocks[4 - level](x)
        
        # final output
        logits = self.final_conv(x)
        return logits

# MULTI-MODALITY FEATURE FUSION
class PerModalityEncoders(nn.Module):
    """
    dedicated encoder for each modality.
    
    processes each modality independently, returning features
    and skip connections for each. handles missing modalities
    with zero tensors.
    """
    def __init__(
        self,
        in_channels: int = 1,
        base_features: int = 32,
        modality_names: Sequence[str] = ALL_MODALITY_NAMES_DEFAULT
    ):
        """
        args:
            in_channels: channels per modality
            base_features: base feature dimension
            modality_names: list of modality names
        """
        super().__init__()
        self.modality_names = list(modality_names)
        self.base_features = base_features
        
        # one encoder per modality
        self.encoders = nn.ModuleDict({
            mod_name: nnUNetEncoder(in_channels, base_features)
            for mod_name in self.modality_names
        })
        
        # store feature dimensions for decoder
        self.encoder_feature_dims = [
            base_features,
            base_features * 2,
            base_features * 4,
            base_features * 8,
            base_features * 16
        ]

    def forward(
        self,
        x_modalities: Dict[str, torch.Tensor]
    ) -> Dict[str, List[torch.Tensor]]:
        """
        encode each modality independently.
        
        args:
            x_modalities: dict mapping modality name to tensor
        
        returns:
            dict mapping modality name to list of skip connections
        """
        modality_features = {}
        
        for mod_name in self.modality_names:
            if mod_name in x_modalities:
                # encode this modality
                skips = self.encoders[mod_name](x_modalities[mod_name])
                modality_features[mod_name] = skips
            else:
                # missing modality: use zero tensors
                batch_size = (
                    next(iter(x_modalities.values())).shape[0]
                )
                device = (
                    next(iter(x_modalities.values())).device
                )
                
                skips = [
                    torch.zeros(
                        batch_size,
                        self.encoder_feature_dims[i],
                        *[(128 // (2 ** i))] * 3,
                        device=device,
                        dtype=torch.float32
                    )
                    for i in range(5)
                ]
                modality_features[mod_name] = skips
        
        return modality_features

class FeatureAggregationModule(nn.Module):
    """
    attention-based multi-modal feature fusion.
    
    learns to weight and fuse features from different modalities
    using both channel attention and spatial attention.
    """
    def __init__(self, feature_dim: int, num_modalities: int = 4):
        """
        args:
            feature_dim: feature channel dimension
            num_modalities: number of modalities to aggregate
        """
        super().__init__()
        self.feature_dim = feature_dim
        self.num_modalities = num_modalities
        
        # modality weighting via channel attention
        self.modality_weighting = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(1),
            nn.Linear(feature_dim * num_modalities, num_modalities),
            nn.Softmax(dim=1)
        )
        
        # spatial attention
        self.spatial_attention = nn.Sequential(
            nn.Conv3d(feature_dim, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(
        self,
        modality_features: List[torch.Tensor]
    ) -> torch.Tensor:
        """
        aggregate multi-modal features via attention.
        
        args:
            modality_features: list of feature tensors, one per modality
        
        returns:
            aggregated feature tensor
        """
        # stack modality features
        stacked = torch.stack(modality_features, dim=1)
        # (batch, num_mods, channels, d, h, w)
        
        batch_size = stacked.shape[0]
        num_mods = stacked.shape[1]
        
        # compute modality weights
        pooled = torch.nn.functional.adaptive_avg_pool3d(
            stacked.view(batch_size, num_mods * self.feature_dim, -1),
            1
        )
        # (batch, num_mods * channels, 1)
        
        mod_weights = self.modality_weighting(
            pooled.view(batch_size, num_mods * self.feature_dim)
        )
        # (batch, num_mods)
        
        # apply modality weights
        weighted = stacked * mod_weights.view(
            batch_size, num_mods, 1, 1, 1, 1
        )
        
        # sum across modalities
        fused = weighted.sum(dim=1)
        # (batch, channels, d, h, w)
        
        # apply spatial attention
        spatial_att = self.spatial_attention(fused)
        fused = fused * spatial_att
        
        return fused

# ADAPTIVE PARAMETER GENERATION
class Adapter(nn.Module):
    """
    lightweight per-voxel bottleneck adapter.
    
    small feed-forward mlp applied residually to feature maps.
    down-projects, applies nonlinearity, up-projects with
    residual connection.
    """
    def __init__(
        self,
        feature_dim: int,
        bottleneck_scale: float = 0.25
    ):
        """
        args:
            feature_dim: input feature dimension
            bottleneck_scale: bottleneck reduction ratio
        """
        super().__init__()
        self.feature_dim = feature_dim
        
        bottleneck_dim = max(
            1,
            int(feature_dim * bottleneck_scale)
        )
        
        # down-project
        self.down_project = nn.Conv3d(
            feature_dim,
            bottleneck_dim,
            kernel_size=1
        )
        
        # activation
        self.activation = nn.ReLU(inplace=True)
        
        # up-project (initialized to near-zero for identity start)
        self.up_project = nn.Conv3d(
            bottleneck_dim,
            feature_dim,
            kernel_size=1
        )
        
        # initialize up-project to near-zero
        nn.init.zeros_(self.up_project.weight)
        nn.init.zeros_(self.up_project.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        apply adapter as residual addition to input.
        
        args:
            x: input tensor of shape (b, c, d, h, w)
        
        returns:
            adapted tensor: x + adapter(x)
        """
        # bottleneck pathway
        adapter_out = self.down_project(x)
        adapter_out = self.activation(adapter_out)
        adapter_out = self.up_project(adapter_out)
        
        # residual connection
        return x + adapter_out

class MetaGenerator(nn.Module):
    """
    generates sample-specific adapter parameters from features.
    
    takes bottleneck features and modality mask, produces
    adapter weight tensors conditioned on sample content.
    enables meta-learning across different data distributions.
    """
    def __init__(
        self,
        bottleneck_dim: int,
        num_levels: int = 5,
        num_modalities: int = 4,
        bottleneck_scale: float = 0.25
    ):
        """
        args:
            bottleneck_dim: encoder bottleneck dimension
            num_levels: number of decoder levels needing adapters
            num_modalities: total number of modalities
            bottleneck_scale: bottleneck reduction ratio for adapters
        """
        super().__init__()
        self.bottleneck_dim = bottleneck_dim
        self.num_levels = num_levels
        self.bottleneck_scale = bottleneck_scale
        
        bottleneck_adapt_dim = max(
            1,
            int(bottleneck_dim * bottleneck_scale)
        )
        
        # encoder: bottleneck + modality mask -> hidden
        self.encoder = nn.Sequential(
            nn.Linear(
                bottleneck_dim + num_modalities,
                256
            ),
            nn.ReLU(inplace=True),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128)
        )
        
        # decoder heads for each level
        self.decoders = nn.ModuleList([
            nn.Sequential(
                nn.Linear(128, 256),
                nn.ReLU(inplace=True),
                nn.Linear(
                    256,
                    (bottleneck_dim * bottleneck_adapt_dim +
                     bottleneck_dim * bottleneck_adapt_dim +
                     bottleneck_adapt_dim * bottleneck_dim +
                     bottleneck_dim)
                )
            )
            for _ in range(num_levels)
        ])

    def forward(
        self,
        bottleneck_features: torch.Tensor,
        modality_mask: torch.Tensor
    ) -> List[Dict[str, torch.Tensor]]:
        """
        generate adapter parameters for all levels.
        
        args:
            bottleneck_features: encoder bottleneck output
                of shape (batch, bottleneck_dim, 1, 1, 1)
            modality_mask: binary modality presence mask
                of shape (batch, num_modalities)
        
        returns:
            list of dicts, one per level, each containing
            adapter parameter tensors
        """
        batch_size = bottleneck_features.shape[0]
        
        # flatten bottleneck
        bottleneck_flat = torch.nn.functional.adaptive_avg_pool3d(
            bottleneck_features,
            1
        ).view(batch_size, self.bottleneck_dim)
        
        # concatenate with modality mask
        combined = torch.cat([bottleneck_flat, modality_mask], dim=1)
        
        # encode
        hidden = self.encoder(combined)
        
        # generate adapter params for each level
        adapter_params_list = []
        
        bottleneck_adapt_dim = max(
            1,
            int(self.bottleneck_dim * self.bottleneck_scale)
        )
        
        for level in range(self.num_levels):
            params = self.decoders[level](hidden)
            
            # split into adapter components
            down_weight_size = (
                bottleneck_adapt_dim * self.bottleneck_dim
            )
            down_bias_size = bottleneck_adapt_dim
            up_weight_size = (
                self.bottleneck_dim * bottleneck_adapt_dim
            )
            
            down_weight = params[
                :, :down_weight_size
            ].view(
                batch_size,
                bottleneck_adapt_dim,
                self.bottleneck_dim,
                1,
                1,
                1
            )
            
            down_bias = params[
                :, down_weight_size:(
                    down_weight_size + down_bias_size
                )
            ].view(batch_size, bottleneck_adapt_dim, 1, 1, 1)
            
            up_weight = params[
                :, (down_weight_size + down_bias_size):(
                    down_weight_size + down_bias_size +
                    up_weight_size
                )
            ].view(
                batch_size,
                self.bottleneck_dim,
                bottleneck_adapt_dim,
                1,
                1,
                1
            )
            
            up_bias = params[
                :, (down_weight_size + down_bias_size +
                    up_weight_size):
            ].view(batch_size, self.bottleneck_dim, 1, 1, 1)
            
            adapter_params_list.append({
                'down_weight': down_weight,
                'down_bias': down_bias,
                'up_weight': up_weight,
                'up_bias': up_bias
            })
        
        return adapter_params_list

# MULTI-MODALITY SEGMENTATION WITH META-GENERATOR
class Generator(nn.Module):
    """
    encoder-decoder generator with per-modality processing
    and meta-learned adaptive parameters.
    
    processes each modality through dedicated encoder,
    aggregates features, optionally generates adaptive
    parameters via meta-generator.
    """
    def __init__(
        self,
        in_channels: int = 1,
        base_features: int = 32,
        all_modality_names: Sequence[str] = ALL_MODALITY_NAMES_DEFAULT,
        use_meta_generator: bool = True,
        adapter_bottleneck_scale: float = 0.25
    ):
        """
        args:
            in_channels: channels per modality
            base_features: base feature dimension
            all_modality_names: list of all modality names
            use_meta_generator: enable meta-generator
            adapter_bottleneck_scale: bottleneck ratio
        """
        super().__init__()
        self.in_channels = in_channels
        self.base_features = base_features
        self.all_modality_names = list(all_modality_names)
        self.use_meta_generator = use_meta_generator
        
        # feature dimensions
        self.encoder_feature_dims = [
            base_features,
            base_features * 2,
            base_features * 4,
            base_features * 8,
            base_features * 16
        ]
        
        # per-modality encoders
        self.per_modality_encoders = PerModalityEncoders(
            in_channels,
            base_features,
            self.all_modality_names
        )
        
        # multi-modality fusion modules (one per level)
        self.aggregators = nn.ModuleList([
            FeatureAggregationModule(
                self.encoder_feature_dims[i],
                len(self.all_modality_names)
            )
            for i in range(5)
        ])
        
        # meta-generator for adaptive parameters
        if self.use_meta_generator:
            self.meta_generator = MetaGenerator(
                self.encoder_feature_dims[4],
                num_levels=5,
                num_modalities=len(self.all_modality_names),
                bottleneck_scale=adapter_bottleneck_scale
            )

    def forward(
        self,
        x_modalities: Dict[str, torch.Tensor],
        modality_mask: Optional[torch.Tensor] = None
    ) -> Tuple[List[torch.Tensor], torch.Tensor]:
        """
        encode multi-modal input with fusion.
        
        args:
            x_modalities: dict of modality tensors
            modality_mask: optional modality presence mask
        
        returns:
            tuple of:
            - fused_skips: list of aggregated skip connections
            - bottleneck_features: encoder bottleneck
        """
        # encode each modality
        modality_features = self.per_modality_encoders(x_modalities)
        
        # aggregate across modalities at each level
        fused_skips = []
        for level in range(5):
            level_features = [
                modality_features[mod_name][level]
                for mod_name in self.all_modality_names
            ]
            
            fused = self.aggregators[level](level_features)
            fused_skips.append(fused)
        
        # bottleneck is the deepest fused feature
        bottleneck_features = fused_skips[4]
        
        return fused_skips, bottleneck_features

# DISCRIMINATOR FOR ADVERSARIAL TRAINING
class ModalityDiscriminator(nn.Module):
    """
    adversarial discriminator predicting modality presence.
    
    takes encoder bottleneck features and predicts which
    modalities are present in the sample. encourages
    modality-invariant feature learning.
    """
    def __init__(
        self,
        bottleneck_dim: int,
        num_modalities: int = 4,
        hidden_dim: int = 256
    ):
        """
        args:
            bottleneck_dim: encoder bottleneck dimension
            num_modalities: number of modality classes
            hidden_dim: hidden layer dimension
        """
        super().__init__()
        self.bottleneck_dim = bottleneck_dim
        self.num_modalities = num_modalities
        
        self.discriminator = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(1),
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_modalities)
        )

    def forward(
        self,
        bottleneck_features: torch.Tensor
    ) -> torch.Tensor:
        """
        predict modality presence from bottleneck.
        
        args:
            bottleneck_features: encoder bottleneck
                of shape (batch, channels, d, h, w)
        
        returns:
            logits of shape (batch, num_modalities)
        """
        return self.discriminator(bottleneck_features)

# MAIN SEGMENTATION MODEL
class SegmentationModel(nn.Module):
    """
    nnunet-based segmentation model with multi-modality fusion.
    integrates dedicated per-modality encoders with attention-based
    feature aggregation and meta-generator for adaptive learning.
    """
    def __init__(
        self,
        in_channels: int = 1,
        out_channels: int = 3,
        base_features: int = 32,
        all_modality_names: Sequence[str] = ALL_MODALITY_NAMES_DEFAULT,
        use_meta_generator: bool = True,
        adapter_bottleneck_scale: float = 0.25
    ):
        """
        args:
            in_channels: input image channels
            out_channels: number of output segmentation classes
            base_features: base feature dimension for encoders
            all_modality_names: names of all modalities
            use_meta_generator: enable meta-generator for adaptive adapters
            adapter_bottleneck_scale: bottleneck ratio for adapters
        """
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.base_features = base_features
        self.all_modality_names = list(all_modality_names)
        self.use_meta_generator = use_meta_generator

        # nnunet generator with per-modality encoders
        self.generator = Generator(
            in_channels=in_channels,
            base_features=base_features,
            all_modality_names=self.all_modality_names,
            use_meta_generator=use_meta_generator,
            adapter_bottleneck_scale=adapter_bottleneck_scale
        )

        # nnunet decoder
        self.decoder = nnUNetDecoder(
            self.generator.encoder_feature_dims,
            out_channels
        )

    def forward(
        self,
        x_modalities: Dict[str, torch.Tensor],
        modality_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor, Optional[List[Dict]]]:
        """
        forward pass through segmentation model.

        args:
            x_modalities: dict of modality tensors
            modality_mask: optional binary modality presence mask

        returns:
            tuple of:
            - segmentation_logits: output predictions
            - bottleneck_features: encoder bottleneck for discriminator
            - adapter_params: list of adapter dicts if meta_generator enabled
        """
        # encode and aggregate modalities
        fused_skips, bottleneck_features = self.generator(
            x_modalities,
            modality_mask=modality_mask
        )

        # get adapter parameters from meta-generator if enabled
        adapter_params = None
        if self.use_meta_generator:
            adapter_params = self.generator.meta_generator(
                bottleneck_features,
                modality_mask if modality_mask is not None else (
                    self._infer_modality_mask(x_modalities)
                )
            )

        # decode fused features to segmentation
        segmentation_logits = self.decoder(fused_skips)

        return segmentation_logits, bottleneck_features, adapter_params

    def _infer_modality_mask(
        self,
        x_modalities: Dict[str, torch.Tensor]
    ) -> torch.Tensor:
        """
        infer modality presence mask from input tensors.

        args:
            x_modalities: dict of modality tensors

        returns:
            binary mask of shape (batch, num_modalities)
        """
        batch_size = next(iter(x_modalities.values())).shape[0]
        device = next(iter(x_modalities.values())).device
        num_mods = len(self.all_modality_names)

        mask = torch.zeros(
            batch_size,
            num_mods,
            device=device,
            dtype=torch.float32
        )

        for i, mod_name in enumerate(self.all_modality_names):
            if mod_name in x_modalities:
                is_present = (
                    torch.sum(
                        x_modalities[mod_name],
                        dim=[1, 2, 3, 4]
                    ) > 0
                ).float()
                mask[:, i] = is_present

        return mask

# BRATS DATASET LOADER
class BraTSDataset(Dataset):
    """
    brain tumor segmentation dataset from brats challenge.
    
    loads multi-modal mri scans and ground truth segmentation,
    applies preprocessing, and returns either full-modality
    or partial-modality samples based on mode.
    """
    def __init__(
        self,
        data_dir: str,
        modality_names: Sequence[str] = ALL_MODALITY_NAMES_DEFAULT,
        target_shape: Tuple[int, int, int] = (128, 128, 128),
        mode: str = "train",
        partial_modality_prob: float = 0.5,
        num_partial_modalities: int = 3
    ):
        """
        args:
            data_dir: path to brats dataset directory
            modality_names: names of modalities in order
            target_shape: target spatial dimensions
            mode: 'train', 'val', or 'test'
            partial_modality_prob: probability of partial modality
            num_partial_modalities: num modalities to keep if partial
        """
        super().__init__()
        self.data_dir = data_dir
        self.modality_names = list(modality_names)
        self.target_shape = target_shape
        self.mode = mode
        self.partial_modality_prob = partial_modality_prob
        self.num_partial_modalities = num_partial_modalities

        # build file list
        self.scans = sorted([
            d for d in os.listdir(data_dir)
            if os.path.isdir(os.path.join(data_dir, d))
        ])

    def __len__(self) -> int:
        """
        return number of scans in dataset.
        
        returns:
            number of scans
        """
        return len(self.scans)

    def __getitem__(
        self,
        idx: int
    ) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, torch.Tensor]:
        """
        load and preprocess scan and segmentation.
        
        args:
            idx: sample index
        
        returns:
            tuple of:
            - modality_dict: dict of modality tensors
            - modality_mask: binary mask of present modalities
            - segmentation: binary segmentation mask
        """
        scan_dir = os.path.join(self.data_dir, self.scans[idx])
        
        # load modalities
        modality_dict = {}
        for mod_name in self.modality_names:
            mod_path = os.path.join(
                scan_dir,
                f"{self.scans[idx]}_{mod_name}.nii.gz"
            )
            
            if os.path.exists(mod_path):
                nifti_obj = nib.load(mod_path)
                volume = np.asarray(nifti_obj.dataobj)
                
                # preprocess
                volume = _preprocess_spatial_utility(
                    volume,
                    self.target_shape
                )
                
                # normalize
                v_min = np.min(volume)
                v_max = np.max(volume)
                if v_max > v_min:
                    volume = (volume - v_min) / (v_max - v_min)
                
                # convert to tensor
                volume = torch.from_numpy(
                    volume[np.newaxis, ...]
                ).float()
                
                modality_dict[mod_name] = volume

        # load segmentation
        seg_path = os.path.join(
            scan_dir,
            f"{self.scans[idx]}_seg.nii.gz"
        )
        seg_volume = np.asarray(nib.load(seg_path).dataobj)
        seg_volume = _preprocess_spatial_utility(
            seg_volume,
            self.target_shape
        )
        seg_mask = _create_multilabel_seg_masks_utility(seg_volume)
        seg_mask = torch.from_numpy(seg_mask).float()

        # create modality mask for present modalities
        num_mods = len(self.modality_names)
        modality_mask = torch.zeros(num_mods, dtype=torch.float32)
        
        for i, mod_name in enumerate(self.modality_names):
            if mod_name in modality_dict:
                modality_mask[i] = 1.0

        # optionally drop some modalities
        if random.random() < self.partial_modality_prob:
            present_mods = [
                i for i, m in enumerate(modality_mask)
                if m > 0
            ]
            
            if len(present_mods) > self.num_partial_modalities:
                keep_idxs = random.sample(
                    present_mods,
                    self.num_partial_modalities
                )
                
                # remove modalities not in keep_idxs
                for i, mod_name in enumerate(self.modality_names):
                    if i not in keep_idxs:
                        modality_dict.pop(mod_name, None)
                        modality_mask[i] = 0.0

        return modality_dict, modality_mask, seg_mask

In [ ]:
# BI-LEVEL META-TRAINER WITH PROPER GRADIENT FLOW
class MetaTrainer:
    """
    bi-level meta-trainer implementing the correct algorithm from the paper.

    inner loop: updates only adapter parameters on partial-modality data.
    outer loop: updates base network and meta-generator on full-modality data
                with gradients flowing through the inner-adapted model.
    both loops: update discriminator for modality-invariant representations.
    """
    def __init__(
        self,
        segmentation_model: SegmentationModel,
        discriminator_model: ModalityDiscriminator,
        dataloader_Dm: DataLoader,
        dataloader_Df: DataLoader,
        device: torch.device,
        lr_meta_G: float = 1e-4,
        lr_inner_adapter: float = 1e-3,
        lr_D: float = 1e-4,
        lambda_seg: float = 1.0,
        lambda_adv_g: float = 0.1,
        lambda_reg_adapter: float = 0.01,
        inner_loop_k_steps: int = 1,
        use_amp: bool = False,
        accumulation_steps: int = 1
    ):
        """
        initialize meta-trainer with bi-level optimization settings.

        args:
            segmentation_model: main segmentation model (base parameters)
            discriminator_model: adversarial modality discriminator
            dataloader_Dm: partial-modality training data loader
            dataloader_Df: full-modality training data loader
            device: torch device (cuda/cpu)
            lr_meta_G: learning rate for base network and meta-generator
            lr_inner_adapter: learning rate for inner loop adapter updates
            lr_D: learning rate for discriminator
            lambda_seg: weight for segmentation loss
            lambda_adv_g: weight for adversarial generator loss
            lambda_reg_adapter: weight for adapter l2 regularization
            inner_loop_k_steps: number of inner loop gradient steps (K)
            use_amp: enable automatic mixed precision
            accumulation_steps: gradient accumulation steps
        """
        self.segmentation_model = segmentation_model.to(device)
        self.discriminator_model = discriminator_model.to(device)
        self.dataloader_Dm = dataloader_Dm
        self.dataloader_Df = dataloader_Df
        self.device = device

        # hyperparameters
        self.lr_inner_adapter = lr_inner_adapter
        self.lambda_seg = lambda_seg
        self.lambda_adv_g = lambda_adv_g
        self.lambda_reg_adapter = lambda_reg_adapter
        self.inner_loop_k_steps = inner_loop_k_steps
        self.use_amp = use_amp if device.type == 'cuda' else False
        self.accumulation_steps = accumulation_steps

        # loss functions
        self.criterion_seg = DiceLoss(
            sigmoid=True,
            include_background=True,
            reduction='mean'
        )
        self.criterion_bce = nn.BCEWithLogitsLoss()

        # optimizers
        self.optimizer_meta = Adam(
            self.segmentation_model.parameters(),
            lr=lr_meta_G
        )
        self.optimizer_D = Adam(
            self.discriminator_model.parameters(),
            lr=lr_D
        )

        # amp gradient scalers
        self.scaler_meta = GradScaler(enabled=self.use_amp)
        self.scaler_D = GradScaler(enabled=self.use_amp)
        self.scaler_inner = GradScaler(enabled=self.use_amp)

        # tracking
        self.epoch_loss_history = {
            "outer_gen": [],
            "outer_seg": [],
            "outer_adv": [],
            "inner_seg": [],
            "inner_adv": [],
            "discriminator": []
        }
        self.epoch_dice_history = []
        self.best_dsc = -1.0

        print(
            f"[MetaTrainer] initialized with bi-level optimization. "
            f"amp={self.use_amp}, k_steps={inner_loop_k_steps}"
        )

    def train_epoch(self, epoch: int):
        """
        perform one training epoch with bi-level meta-learning.

        implements algorithm 2 from paper:
        - inner loop: adapt Δθ on partial-modality data D_m
        - outer loop: update θ, φ on full-modality data D_f
        - both loops: update discriminator D_ψ

        args:
            epoch: current epoch number
        """
        self.segmentation_model.train()
        self.discriminator_model.train()

        iter_Df = iter(self.dataloader_Df)
        total_batches = len(self.dataloader_Dm)

        # loss tracking
        losses = {
            "inner_seg": [], "inner_adv": [],
            "outer_seg": [], "outer_adv": [],
            "discrim": []
        }
        metrics = {"dice": [], "hd": []}

        print(f"\n> Epoch {epoch + 1}")

        with tqdm(
            enumerate(self.dataloader_Dm),
            total=total_batches,
            desc=f"Epoch {epoch + 1}"
        ) as pbar:
            for batch_idx, (batch_Dm, mask_Dm, seg_Dm) in pbar:
                # move to device
                batch_Dm = {k: v.to(self.device)
                           for k, v in batch_Dm.items()}
                mask_Dm = mask_Dm.to(self.device)
                seg_Dm = seg_Dm.to(self.device)
                batch_size = seg_Dm.shape[0]

                # INNER LOOP: Adapt Δθ on partial-modality data
                # create functional model for inner loop
                inner_model = copy.deepcopy(
                    self.segmentation_model
                )
                inner_model = inner_model.to(self.device)

                # optimizer for inner loop
                if self.segmentation_model.use_meta_generator:
                    # only adapt meta-generator
                    inner_params = list(
                        inner_model.generator.meta_generator.parameters()
                    )
                else:
                    inner_params = list(inner_model.parameters())

                optimizer_inner = Adam(
                    inner_params,
                    lr=self.lr_inner_adapter
                )

                # K-step inner loop
                for k in range(self.inner_loop_k_steps):
                    optimizer_inner.zero_grad()

                    with torch.amp.autocast(
                        self.device.type,
                        enabled=self.use_amp
                    ):
                        # forward through adapted model
                        seg_logits_inner, bottleneck_inner, _ = (
                            inner_model(batch_Dm, modality_mask=mask_Dm)
                        )

                        # discriminator prediction
                        d_logits_inner = (
                            self.discriminator_model(bottleneck_inner)
                        )

                        # segmentation loss
                        loss_seg_inner = self.criterion_seg(
                            seg_logits_inner,
                            seg_Dm
                        )

                        # adversarial generator loss
                        target_all_present = torch.ones_like(
                            d_logits_inner,
                            device=self.device
                        )
                        loss_adv_inner = self.criterion_bce(
                            d_logits_inner,
                            target_all_present
                        )

                        # combined inner loss
                        loss_inner = (
                            self.lambda_seg * loss_seg_inner +
                            self.lambda_adv_g * loss_adv_inner
                        )

                    # backprop through inner model
                    self.scaler_inner.scale(loss_inner).backward()
                    self.scaler_inner.step(optimizer_inner)
                    self.scaler_inner.update()

                    losses["inner_seg"].append(loss_seg_inner.item())
                    losses["inner_adv"].append(loss_adv_inner.item())

                # update discriminator on inner loop partial data
                self.optimizer_D.zero_grad()
                with torch.no_grad(), torch.amp.autocast(
                    self.device.type,
                    enabled=self.use_amp
                ):
                    _, bottleneck_for_D, _ = (
                        inner_model(batch_Dm, modality_mask=mask_Dm)
                    )

                with torch.amp.autocast(
                    self.device.type,
                    enabled=self.use_amp
                ):
                    d_logits_for_D = (
                        self.discriminator_model(bottleneck_for_D.detach())
                    )
                    loss_D_inner = self.criterion_bce(
                        d_logits_for_D,
                        mask_Dm
                    )

                self.scaler_D.scale(loss_D_inner).backward()
                self.scaler_D.step(self.optimizer_D)
                self.scaler_D.update()

                # OUTER LOOP: Update θ, φ on full-modality data
                # get next batch from full-modality dataloader
                try:
                    batch_Df, mask_Df, seg_Df = next(iter_Df)
                except StopIteration:
                    iter_Df = iter(self.dataloader_Df)
                    batch_Df, mask_Df, seg_Df = next(iter_Df)

                batch_Df = {k: v.to(self.device)
                           for k, v in batch_Df.items()}
                mask_Df = mask_Df.to(self.device)
                seg_Df = seg_Df.to(self.device)

                # compute outer loss through adapted model
                # gradients flow from outer_loss back through
                # inner_model to main_model base parameters
                self.optimizer_meta.zero_grad()

                with torch.amp.autocast(
                    self.device.type,
                    enabled=self.use_amp
                ):
                    # forward on adapted model from inner loop
                    seg_logits_outer, bottleneck_outer, adapter_params = (
                        inner_model(batch_Df, modality_mask=mask_Df)
                    )

                    # discriminator on full-modality data
                    d_logits_outer = (
                        self.discriminator_model(bottleneck_outer)
                    )

                    # segmentation loss
                    loss_seg_outer = self.criterion_seg(
                        seg_logits_outer,
                        seg_Df
                    )

                    # adversarial loss
                    loss_adv_outer = self.criterion_bce(
                        d_logits_outer,
                        target_all_present
                    )

                    # adapter regularization
                    loss_reg = torch.tensor(0.0, device=self.device)
                    if (self.segmentation_model.use_meta_generator and
                        adapter_params is not None):
                        for adapter_dict in adapter_params:
                            for param_tensor in adapter_dict.values():
                                loss_reg = loss_reg + (
                                    self.lambda_reg_adapter *
                                    torch.norm(param_tensor, p=2)
                                )

                    # combined outer loss
                    loss_outer = (
                        self.lambda_seg * loss_seg_outer +
                        self.lambda_adv_g * loss_adv_outer +
                        loss_reg
                    )

                # backprop through adapted inner model to main model
                # KEY FIX: gradients flow to base θ and φ parameters
                loss_outer_scaled = loss_outer / self.accumulation_steps
                self.scaler_meta.scale(loss_outer_scaled).backward()

                if (batch_idx + 1) % self.accumulation_steps == 0:
                    self.scaler_meta.step(self.optimizer_meta)
                    self.scaler_meta.update()
                    self.optimizer_meta.zero_grad()

                # update discriminator on outer loop full-modality data
                self.optimizer_D.zero_grad()
                with torch.no_grad(), torch.amp.autocast(
                    self.device.type,
                    enabled=self.use_amp
                ):
                    _, bottleneck_for_D_outer, _ = (
                        inner_model(batch_Df, modality_mask=mask_Df)
                    )

                with torch.amp.autocast(
                    self.device.type,
                    enabled=self.use_amp
                ):
                    d_logits_for_D_outer = (
                        self.discriminator_model(
                            bottleneck_for_D_outer.detach()
                        )
                    )
                    loss_D_outer = self.criterion_bce(
                        d_logits_for_D_outer,
                        mask_Df
                    )

                loss_D_outer_scaled = loss_D_outer / self.accumulation_steps
                self.scaler_D.scale(loss_D_outer_scaled).backward()

                if (batch_idx + 1) % self.accumulation_steps == 0:
                    self.scaler_D.step(self.optimizer_D)
                    self.scaler_D.update()
                    self.optimizer_D.zero_grad()

                # track losses
                losses["outer_seg"].append(loss_seg_outer.item())
                losses["outer_adv"].append(loss_adv_outer.item())
                losses["discriminator"].append(
                    (loss_D_inner.item() + loss_D_outer.item()) / 2
                )

                # compute metrics
                with torch.no_grad():
                    seg_probs = torch.sigmoid(seg_logits_outer)
                    seg_preds = (seg_probs > 0.5).float()

                    dice = DiceMetric(include_background=False)(
                        seg_preds,
                        seg_Df
                    )
                    metrics["dice"].append(dice)

                # update progress bar
                if losses["outer_seg"]:
                    mean_seg_loss = sum(
                        losses["outer_seg"]
                    ) / len(losses["outer_seg"])
                    mean_dice = (
                        sum(m.item() if torch.is_tensor(m) else m
                            for m in metrics["dice"]) /
                        len(metrics["dice"])
                        if metrics["dice"] else 0.0
                    )
                    pbar.set_postfix({
                        "seg_loss": f"{mean_seg_loss:.4f}",
                        "adv_loss": f"{losses['outer_adv'][-1]:.4f}",
                        "dice": f"{mean_dice:.4f}"
                    })

        # epoch summary
        if losses["outer_seg"]:
            epoch_seg = sum(losses["outer_seg"]) / len(losses["outer_seg"])
            epoch_adv = sum(losses["outer_adv"]) / len(losses["outer_adv"])
            epoch_dice = (
                sum(m.item() if torch.is_tensor(m) else m
                    for m in metrics["dice"]) /
                len(metrics["dice"])
                if metrics["dice"] else 0.0
            )

            self.epoch_loss_history["outer_seg"].append(epoch_seg)
            self.epoch_loss_history["outer_adv"].append(epoch_adv)
            self.epoch_dice_history.append(epoch_dice)

            # save if improved
            if epoch_dice > self.best_dsc:
                old_best = self.best_dsc
                self.best_dsc = epoch_dice
                torch.save({
                    "model": self.segmentation_model.state_dict(),
                    "discriminator": self.discriminator_model.state_dict(),
                    "epoch": epoch,
                    "best_dice": self.best_dsc
                }, "best_model.pth")

                if old_best > 0:
                    improvement = 100 * (self.best_dsc - old_best) / old_best
                    print(f"[Epoch {epoch + 1}] Dice improved "
                          f"{improvement:.2f}% "
                          f"({old_best:.4f} → {self.best_dsc:.4f})")

    def train(self, num_epochs: int):
        """
        run training for specified number of epochs.

        args:
            num_epochs: number of epochs to train
        """
        for epoch in range(num_epochs):
            self.train_epoch(epoch)

        print("\n[Training Complete] Bi-level meta-learning finished.")

# INITIAL TRAINING

In [ ]:
# if __name__ == '__main__':

#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Using device: {device}")

#     USE_AMP = True if device.type == 'cuda' else False
#     ACCUMULATION_STEPS = 2
#     INPUT_CHANNELS = 1
#     SEG_OUTPUT_CHANNELS = 3  # Changed to 3 for proper BraTS evaluation
#     LOAD_SEGMENTATION_MASKS = True
#     USE_ADAPTERS = True
#     BATCH_SIZE_DM = 1  # Reduced due to larger image size
#     BATCH_SIZE_DF = 1  # Reduced due to larger image size

#     # Define consistent target image size for all processing
#     IMG_DIM_TARGET = (128, 128, 128)  # Consistent target size for nnU-Net
#     MODEL_FEATURE_SIZE = 32  # Changed to match nnU-Net base features

#     ALL_MODS_TUPLE = ("flair", "t1", "t1ce", "t2")
#     NUM_MODALITIES_TOTAL = len(ALL_MODS_TUPLE)
#     NUM_PATIENTS_PARTIAL_DM = 200
#     NUM_PATIENTS_FULL_DF = 50
#     NUM_MODS_SELECT_PARTIAL = 3
#     INNER_LOOP_LR_ALPHA = 1e-3
#     INNER_LOOP_K_STEPS = 1
#     META_LR_G = 1e-4
#     LR_D = 1e-4
#     LAMBDA_SEG = 1.0
#     LAMBDA_ADV_G = 0.01
#     NUM_EPOCHS_META = 30

#     dataset_parent_dir = "/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
#     print(f"INFO: loadig data from: {dataset_parent_dir}")

#     if not os.path.exists(dataset_parent_dir) and dataset_parent_dir.startswith("/path/to/your"):
#         print("WARNING: Placeholder dataset path used. You must update 'dataset_parent_dir' to run the script. Skipping training and evaluation.")
#     elif not os.path.exists(dataset_parent_dir):
#         print(f"ERROR: Dataset directory not found: {dataset_parent_dir}. Please check the path. Skipping training and evaluation.")
#     else:
#         dataset_Dm = BraTSDataset(
#             data_root_dir=dataset_parent_dir, num_patients_to_load=NUM_PATIENTS_PARTIAL_DM,
#             mode='partial_Dm', num_modalities_to_select_partial=NUM_MODS_SELECT_PARTIAL,
#             target_img_size=IMG_DIM_TARGET,  # Use consistent target size
#             all_possible_modality_keys=list(ALL_MODS_TUPLE),
#             image_output_channels=INPUT_CHANNELS, seg_output_channels=SEG_OUTPUT_CHANNELS,
#             load_segmentation=LOAD_SEGMENTATION_MASKS
#         )
#         dataset_Df = BraTSDataset(
#             data_root_dir=dataset_parent_dir, num_patients_to_load=NUM_PATIENTS_FULL_DF,
#             mode='full_Df', target_img_size=IMG_DIM_TARGET,  # Use consistent target size
#             all_possible_modality_keys=list(ALL_MODS_TUPLE),
#             image_output_channels=INPUT_CHANNELS, seg_output_channels=SEG_OUTPUT_CHANNELS,
#             load_segmentation=LOAD_SEGMENTATION_MASKS
#         )

#         if len(dataset_Dm) > 0 and (len(dataset_Df) > 0 if BATCH_SIZE_DF > 0 else True):
#             dataloader_Dm = DataLoader(dataset_Dm, batch_size=BATCH_SIZE_DM, shuffle=True, num_workers=2)
#             dataloader_Df = DataLoader(dataset_Df, batch_size=BATCH_SIZE_DF, shuffle=True, num_workers=2) if BATCH_SIZE_DF > 0 else None

#             segmentation_model = SegmentationModel(
#                 in_channels=INPUT_CHANNELS,
#                 out_channels=SEG_OUTPUT_CHANNELS,
#                 base_features=MODEL_FEATURE_SIZE,
#                 all_possible_modality_names=ALL_MODS_TUPLE,
#                 use_adapters_in_generator=USE_ADAPTERS
#             )

#             bottleneck_dim = segmentation_model.generator.encoder_feature_dims[-1]
#             discriminator_model = ModalityDiscriminator(bottleneck_feature_dim=bottleneck_dim, num_modalities=NUM_MODALITIES_TOTAL)

#             meta_trainer = MetaTrainer(
#                 segmentation_model=segmentation_model, discriminator_model=discriminator_model,
#                 dataloader_Dm=dataloader_Dm, dataloader_Df=dataloader_Df, device=device,
#                 lr_meta_G=META_LR_G, lr_D=LR_D, lr_alpha_inner=INNER_LOOP_LR_ALPHA,
#                 lambda_seg=LAMBDA_SEG if LOAD_SEGMENTATION_MASKS else 0.0,
#                 lambda_adv_g=LAMBDA_ADV_G,
#                 seg_output_channels_for_loss=SEG_OUTPUT_CHANNELS if LOAD_SEGMENTATION_MASKS else 0,
#                 inner_loop_k_steps=INNER_LOOP_K_STEPS,
#                 use_amp=USE_AMP,
#                 accumulation_steps=ACCUMULATION_STEPS
#             )

#             meta_trainer.train(num_epochs=NUM_EPOCHS_META)


#             test_patient_id = os.path.basename(dataset_Dm.patient_folders[0])
#             test_image_files = {mod_key: os.path.join(dataset_Dm.patient_folders[0], f"{test_patient_id}{dataset_Dm.modality_file_suffixes[mod_key]}") for mod_key in ALL_MODS_TUPLE}
#             test_seg_file = os.path.join(dataset_Dm.patient_folders[0], f"{test_patient_id}{dataset_Dm.seg_file_suffix}")

#             meta_trainer.evaluate_and_plot(
#                 image_file_paths=test_image_files,
#                 ground_truth_seg_path=test_seg_file,
#                 patient_id=test_patient_id
#             )

#             test_patient_id = os.path.basename(dataset_Df.patient_folders[0])
#             test_image_files = {mod_key: os.path.join(dataset_Df.patient_folders[0], f"{test_patient_id}{dataset_Df.modality_file_suffixes[mod_key]}") for mod_key in ALL_MODS_TUPLE}
#             test_seg_file = os.path.join(dataset_Df.patient_folders[0], f"{test_patient_id}{dataset_Df.seg_file_suffix}")

#             meta_trainer.evaluate_and_plot(
#                 image_file_paths=test_image_files,
#                 ground_truth_seg_path=test_seg_file,
#                 patient_id=test_patient_id
#             )
#         else:
#             print(f"WARNING: No valid data found for training with the given parameters.")

# RESUME TRAINING

In [ ]:
# Define the path to your saved model
saved_model_path = "/kaggle/input/metal-b-k/best_model.pth"
print(f"INFO: Attempting to load model from: {saved_model_path}")

if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"INFO: Using device: {device}")

    USE_AMP = True if device.type == 'cuda' else False
    ACCUMULATION_STEPS = 2
    INPUT_CHANNELS = 1
    SEG_OUTPUT_CHANNELS = 3  # Changed to 3 for proper BraTS evaluation
    LOAD_SEGMENTATION_MASKS = True
    USE_ADAPTERS = True
    BATCH_SIZE_DM = 1  # Reduced due to larger image size
    BATCH_SIZE_DF = 1  # Reduced due to larger image size

    # Define consistent target image size for all processing
    IMG_DIM_TARGET = (128, 128, 128)  # Consistent target size for nnU-Net

    MODEL_FEATURE_SIZE = 32  # Changed to match nnU-Net base features
    ALL_MODS_TUPLE = ("flair", "t1", "t1ce", "t2")
    NUM_MODALITIES_TOTAL = len(ALL_MODS_TUPLE)
    NUM_PATIENTS_PARTIAL_DM = 160
    NUM_PATIENTS_FULL_DF = 40
    NUM_MODS_SELECT_PARTIAL = 3
    INNER_LOOP_LR_ALPHA = 1e-3
    INNER_LOOP_K_STEPS = 1
    META_LR_G = 1e-4
    LR_D = 1e-4
    LAMBDA_SEG = 1.0
    LAMBDA_ADV_G = 0.01
    NUM_EPOCHS_META = 90

    dataset_parent_dir = "/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
    print(f"INFO: Attempting to load data from: {dataset_parent_dir}")

    if not os.path.exists(dataset_parent_dir) and dataset_parent_dir.startswith("/path/to/your"):
        print("WARNING: Placeholder dataset path used. You must update 'dataset_parent_dir' to run the script. Skipping training and evaluation.")
    elif not os.path.exists(dataset_parent_dir):
        print(f"ERROR: Dataset directory not found: {dataset_parent_dir}. Please check the path. Skipping training and evaluation.")
    else:
        dataset_Dm = BraTSDataset(
            data_root_dir=dataset_parent_dir, num_patients_to_load=NUM_PATIENTS_PARTIAL_DM,
            mode='partial_Dm', num_modalities_to_select_partial=NUM_MODS_SELECT_PARTIAL,
            target_img_size=IMG_DIM_TARGET,
            all_possible_modality_keys=list(ALL_MODS_TUPLE),
            image_output_channels=INPUT_CHANNELS, seg_output_channels=SEG_OUTPUT_CHANNELS,
            load_segmentation=LOAD_SEGMENTATION_MASKS
        )
        dataset_Df = BraTSDataset(
            data_root_dir=dataset_parent_dir, num_patients_to_load=NUM_PATIENTS_FULL_DF,
            mode='full_Df', target_img_size=IMG_DIM_TARGET,
            all_possible_modality_keys=list(ALL_MODS_TUPLE),
            image_output_channels=INPUT_CHANNELS, seg_output_channels=SEG_OUTPUT_CHANNELS,
            load_segmentation=LOAD_SEGMENTATION_MASKS
        )

        if len(dataset_Dm) > 0 and (len(dataset_Df) > 0 if BATCH_SIZE_DF > 0 else True):
            dataloader_Dm = DataLoader(dataset_Dm, batch_size=BATCH_SIZE_DM, shuffle=True, num_workers=0)
            dataloader_Df = DataLoader(dataset_Df, batch_size=BATCH_SIZE_DF, shuffle=True, num_workers=0) if BATCH_SIZE_DF > 0 else None

            # Create models
            segmentation_model = SegmentationModel(
                in_channels=INPUT_CHANNELS,
                out_channels=SEG_OUTPUT_CHANNELS,
                base_features=MODEL_FEATURE_SIZE,
                all_possible_modality_names=ALL_MODS_TUPLE,
                use_adapters_in_generator=USE_ADAPTERS
            )

            bottleneck_dim = segmentation_model.generator.encoder_feature_dims[-1]
            discriminator_model = ModalityDiscriminator(
                bottleneck_feature_dim=bottleneck_dim,
                num_modalities=NUM_MODALITIES_TOTAL
            )

            # Load saved model if it exists
            saved_model_path = "/kaggle/input/metal-b-k/best_model.pth"
            start_epoch = 30
            if os.path.exists(saved_model_path):
                checkpoint = torch.load(saved_model_path)
                segmentation_model.load_state_dict(checkpoint["segmentation_model"])
                discriminator_model.load_state_dict(checkpoint["discriminator_model"])
                start_epoch = checkpoint["epoch"] + 1  # Start from next epoch
                print(f"INFO: Loaded saved model from epoch {start_epoch-1}, DSC {checkpoint['best_dsc']:.4f}")
                print(f"INFO: Continuing training from epoch {start_epoch}")

            meta_trainer = MetaTrainer(
                segmentation_model=segmentation_model,
                discriminator_model=discriminator_model,
                dataloader_Dm=dataloader_Dm,
                dataloader_Df=dataloader_Df,
                device=device,
                lr_meta_G=META_LR_G,
                lr_D=LR_D,
                lr_alpha_inner=INNER_LOOP_LR_ALPHA,
                lambda_seg=LAMBDA_SEG if LOAD_SEGMENTATION_MASKS else 0.0,
                lambda_adv_g=LAMBDA_ADV_G,
                seg_output_channels_for_loss=SEG_OUTPUT_CHANNELS if LOAD_SEGMENTATION_MASKS else 0,
                inner_loop_k_steps=INNER_LOOP_K_STEPS,
                use_amp=USE_AMP,
                accumulation_steps=ACCUMULATION_STEPS
            )

            # Continue training for additional epochs
            additional_epochs = 20  # Train for 20 more epochs
            total_epochs = start_epoch + additional_epochs
            print(f"INFO: Training for {additional_epochs} additional epochs (total: {total_epochs} epochs)")
            meta_trainer.train(num_epochs=additional_epochs)

            # Evaluation on sample patients
            test_patient_id = os.path.basename(dataset_Dm.patient_folders[0])
            test_image_files = {
                mod_key: os.path.join(dataset_Dm.patient_folders[0],
                f"{test_patient_id}{dataset_Dm.modality_file_suffixes[mod_key]}")
                for mod_key in ALL_MODS_TUPLE
            }
            test_seg_file = os.path.join(
                dataset_Dm.patient_folders[0],
                f"{test_patient_id}{dataset_Dm.seg_file_suffix}"
            )

            meta_trainer.evaluate_and_plot(
                image_file_paths=test_image_files,
                ground_truth_seg_path=test_seg_file,
                patient_id=test_patient_id
            )

            test_patient_id = os.path.basename(dataset_Df.patient_folders[0])
            test_image_files = {
                mod_key: os.path.join(dataset_Df.patient_folders[0],
                f"{test_patient_id}{dataset_Df.modality_file_suffixes[mod_key]}")
                for mod_key in ALL_MODS_TUPLE
            }
            test_seg_file = os.path.join(
                dataset_Df.patient_folders[0],
                f"{test_patient_id}{dataset_Df.seg_file_suffix}"
            )

            meta_trainer.evaluate_and_plot(
                image_file_paths=test_image_files,
                ground_truth_seg_path=test_seg_file,
                patient_id=test_patient_id
            )
        else:
            print(f"WARNING: No valid data found for training with the given parameters.")

# EVALUATION

In [ ]:
# Define the path to your saved model
model_save_path = "/kaggle/input/model-weight/best_model.pth"
print(f"INFO: Attempting to load model from: {model_save_path}")

# Define model parameters (should match training config)
INPUT_CHANNELS = 1
SEG_OUTPUT_CHANNELS = 3
MODEL_FEATURE_SIZE = 32  # Must match what you used in training
ALL_MODS_TUPLE = ("flair", "t1", "t1ce", "t2")
USE_ADAPTERS = True
BATCH_SIZE_DF = 1  # Match your training setting

# Initialize model architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the saved model
if os.path.exists(model_save_path):
    try:
        checkpoint = torch.load(model_save_path, map_location=device)

        # Initialize models
        segmentation_model_loaded = SegmentationModel(
            in_channels=INPUT_CHANNELS,
            out_channels=SEG_OUTPUT_CHANNELS,
            base_features=MODEL_FEATURE_SIZE,
            all_possible_modality_names=ALL_MODS_TUPLE,
            use_adapters_in_generator=USE_ADAPTERS
        ).to(device)

        segmentation_model_loaded.load_state_dict(checkpoint["segmentation_model"])
        print(f"Successfully loaded segmentation model from {model_save_path}")

        # Initialize a minimal discriminator just for evaluation compatibility
        dummy_discriminator = ModalityDiscriminator(
            bottleneck_feature_dim=segmentation_model_loaded.generator.encoder_feature_dims[-1],
            num_modalities=len(ALL_MODS_TUPLE)
        ).to(device)

        # Create a temporary trainer for evaluation
        class EvalTrainer:
            def __init__(self, segmentation_model, device):
                self.segmentation_model = segmentation_model
                self.device = device
                self.epoch_loss_history = {}
                self.epoch_dice_history = []
                self.epoch_hd_history = []

            def evaluate_metrics(self, dataloader, device):
                self.segmentation_model.eval()
                metrics = {
                    'WT': {'Dice': [], 'Accuracy': [], 'Precision': [],
                          'Recall': [], 'F1': [], 'Sensitivity': [], 'Specificity': []},
                    'TC': {'Dice': [], 'Accuracy': [], 'Precision': [],
                          'Recall': [], 'F1': [], 'Sensitivity': [], 'Specificity': []},
                    'ET': {'Dice': [], 'Accuracy': [], 'Precision': [],
                          'Recall': [], 'F1': [], 'Sensitivity': [], 'Specificity': []}
                }

                with torch.no_grad():
                    for batch_data, _, seg_gt in tqdm(dataloader, desc="Evaluating"):
                        if seg_gt is None:
                            continue

                        batch_data = {k: v.to(device) for k, v in batch_data.items()}
                        seg_gt = seg_gt.to(device)

                        seg_logits, _ = self.segmentation_model(batch_data)
                        seg_pred = (torch.sigmoid(seg_logits) > 0.5).float()

                        if seg_pred.shape[2:] != seg_gt.shape[2:]:
                            seg_gt = F.interpolate(seg_gt, size=seg_pred.shape[2:], mode='nearest')

                        for i, name in enumerate(['WT', 'TC', 'ET']):
                            pred = seg_pred[:, i]
                            gt = seg_gt[:, i]

                            # Calculate Dice
                            intersection = (pred * gt).sum()
                            union = pred.sum() + gt.sum()
                            dice = (2 * intersection) / (union + 1e-7)
                            metrics[name]['Dice'].append(dice.item())

                            # Calculate other metrics
                            pred = pred.flatten()
                            gt = gt.flatten()
                            tp = (pred * gt).sum()
                            fp = (pred * (1 - gt)).sum()
                            tn = ((1 - pred) * (1 - gt)).sum()
                            fn = ((1 - pred) * gt).sum()

                            eps = 1e-7
                            metrics[name]['Accuracy'].append(((tp + tn) / (tp + tn + fp + fn + eps)).item())
                            metrics[name]['Precision'].append((tp / (tp + fp + eps)).item())
                            metrics[name]['Recall'].append((tp / (tp + fn + eps)).item())
                            metrics[name]['F1'].append((2 * tp / (2 * tp + fp + fn + eps)).item())
                            metrics[name]['Sensitivity'].append((tp / (tp + fn + eps)).item())
                            metrics[name]['Specificity'].append((tn / (tn + fp + eps)).item())

                # Compute mean metrics
                final_metrics = {}
                for name in metrics:
                    final_metrics[name] = {
                        # 'Dice': np.mean(metrics[name]['Dice']),
                        'Accuracy': np.mean(metrics[name]['Accuracy']),
                        # 'Precision': np.mean(metrics[name]['Precision']),
                        # 'Recall': np.mean(metrics[name]['Recall']),
                        # 'F1': np.mean(metrics[name]['F1']),
                        # 'Sensitivity': np.mean(metrics[name]['Sensitivity']),
                        'Specificity': np.mean(metrics[name]['Specificity'])
                    }

                return final_metrics

        eval_trainer = EvalTrainer(segmentation_model_loaded, device)

        # --- Evaluate Metrics ---
        if 'dataset_Df' in locals():
            eval_dataloader = DataLoader(dataset_Df, batch_size=1, shuffle=False)
            metrics = eval_trainer.evaluate_metrics(eval_dataloader, device)

            # Pretty print metrics
            import pandas as pd
            print(pd.DataFrame(metrics).T.round(4))
        else:
            print("\nWARNING: dataset_Df not found - can't compute metrics")

    except Exception as e:
        print(f"ERROR during evaluation: {str(e)}")
        raise e  # Re-raise to see full traceback
else:
    print(f"Model file not found at {model_save_path}")

INFO: Attempting to load model from: /kaggle/input/model-weight/best_model.pth
Successfully loaded segmentation model from /kaggle/input/model-weight/best_model.pth


Evaluating: 100%|██████████| 40/40 [00:57<00:00,  1.43s/it]

    Accuracy  Specificity
WT    0.9830       0.9953
TC    0.9866       0.9911
ET    0.9872       0.9897


In [ ]:
# Define the path to your saved model
model_save_path = "/kaggle/input/model-weight/best_model.pth"
print(f"INFO: Attempting to load model from: {model_save_path}")

# Define model parameters (should match training config)
INPUT_CHANNELS = 1
SEG_OUTPUT_CHANNELS = 3
MODEL_FEATURE_SIZE = 32  # Must match what you used in training
ALL_MODS_TUPLE = ("flair", "t1", "t1ce", "t2")
USE_ADAPTERS = True
BATCH_SIZE_DF = 1  # Match your training setting

# Initialize model architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the saved model
if os.path.exists(model_save_path):
    try:
        checkpoint = torch.load(model_save_path, map_location=device)

        # Initialize models
        segmentation_model_loaded = SegmentationModel(
            in_channels=INPUT_CHANNELS,
            out_channels=SEG_OUTPUT_CHANNELS,
            base_features=MODEL_FEATURE_SIZE,
            all_possible_modality_names=ALL_MODS_TUPLE,
            use_adapters_in_generator=USE_ADAPTERS
        ).to(device)

        segmentation_model_loaded.load_state_dict(checkpoint["segmentation_model"])
        print(f"Successfully loaded segmentation model from {model_save_path}")

        # Initialize a minimal discriminator just for evaluation compatibility
        dummy_discriminator = ModalityDiscriminator(
            bottleneck_feature_dim=segmentation_model_loaded.generator.encoder_feature_dims[-1],
            num_modalities=len(ALL_MODS_TUPLE)
        ).to(device)

        # Create a temporary trainer for evaluation
        class EvalTrainer:
            def __init__(self, segmentation_model, device):
                self.segmentation_model = segmentation_model
                self.device = device
                self.epoch_loss_history = {}
                self.epoch_dice_history = []
                self.epoch_hd_history = []

            def evaluate_metrics(self, dataloader, device):
                self.segmentation_model.eval()
                metrics = {
                    'WT': {'Dice': [], 'Accuracy': [], 'Precision': [],
                          'Recall': [], 'F1': [], 'Sensitivity': [], 'Specificity': []},
                    'TC': {'Dice': [], 'Accuracy': [], 'Precision': [],
                          'Recall': [], 'F1': [], 'Sensitivity': [], 'Specificity': []},
                    'ET': {'Dice': [], 'Accuracy': [], 'Precision': [],
                          'Recall': [], 'F1': [], 'Sensitivity': [], 'Specificity': []}
                }

                with torch.no_grad():
                    for batch_data, _, seg_gt in tqdm(dataloader, desc="Evaluating"):
                        if seg_gt is None:
                            continue

                        batch_data = {k: v.to(device) for k, v in batch_data.items()}
                        seg_gt = seg_gt.to(device)

                        seg_logits, _ = self.segmentation_model(batch_data)
                        seg_pred = (torch.sigmoid(seg_logits) > 0.5).float()

                        if seg_pred.shape[2:] != seg_gt.shape[2:]:
                            seg_gt = F.interpolate(seg_gt, size=seg_pred.shape[2:], mode='nearest')

                        for i, name in enumerate(['WT', 'TC', 'ET']):
                            pred = seg_pred[:, i]
                            gt = seg_gt[:, i]

                            # Calculate Dice
                            intersection = (pred * gt).sum()
                            union = pred.sum() + gt.sum()
                            dice = (2 * intersection) / (union + 1e-7)
                            metrics[name]['Dice'].append(dice.item())

                            # Calculate other metrics
                            pred = pred.flatten()
                            gt = gt.flatten()
                            tp = (pred * gt).sum()
                            fp = (pred * (1 - gt)).sum()
                            tn = ((1 - pred) * (1 - gt)).sum()
                            fn = ((1 - pred) * gt).sum()

                            eps = 1e-7
                            metrics[name]['Accuracy'].append(((tp + tn) / (tp + tn + fp + fn + eps)).item())
                            metrics[name]['Specificity'].append((tn / (tn + fp + eps)).item())

                # Compute mean metrics
                final_metrics = {}
                for name in metrics:
                    final_metrics[name] = {
                        'Dice': np.mean(metrics[name]['Dice']),
                        'Accuracy': np.mean(metrics[name]['Accuracy']),
                        'Specificity': np.mean(metrics[name]['Specificity'])
                    }

                return final_metrics

        eval_trainer = EvalTrainer(segmentation_model_loaded, device)

        # --- Evaluate Metrics ---
        if 'dataset_Df' in locals():
            eval_dataloader = DataLoader(dataset_Df, batch_size=1, shuffle=False)
            metrics = eval_trainer.evaluate_metrics(eval_dataloader, device)

            # Pretty print metrics
            import pandas as pd
            print(pd.DataFrame(metrics).T.round(4))
        else:
            print("\nWARNING: dataset_Df not found - can't compute metrics")

    except Exception as e:
        print(f"ERROR during evaluation: {str(e)}")
        raise e  # Re-raise to see full traceback
else:
    print(f"Model file not found at {model_save_path}")

INFO: Attempting to load model from: /kaggle/input/model-weight/best_model.pth
Successfully loaded segmentation model from /kaggle/input/model-weight/best_model.pth


Evaluating: 100%|██████████| 40/40 [00:56<00:00,  1.42s/it]

     Dice  Accuracy  Specificity
WT  92.59      0.98         1.00
TC  90.86      0.99         0.99
ET  87.12      0.99         0.99
